# Notebook 0: 汇总各简单运动场景的最优参数集

本 notebook 扫描所有运动场景的贝叶斯优化结果，汇总为标准化的参数字典。

**前置条件**: 跳绳/俯卧撑/手臂弯举/开合跳场景的 `batch_outputs` 已通过 GUI 或 CLI 生成。

In [ ]:
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"D:\data\PPG_HeartRate\Algorithm\outline-PPGtoHR")
DATA_DIR = PROJECT_ROOT / "20260418test_python"
ARTIFACTS_DIR = PROJECT_ROOT / "docs" / "research" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / "python" / "src"))

from ppg_hr.params import SolverParams

## 1. 定义运动场景和文件映射

In [ ]:
EXERCISES = {
    "jump_rope": {
        "dir": "tiaosheng",
        "label": "跳绳",
        "complexity": "simple",
    },
    "arm_curl": {
        "dir": "wanju",
        "label": "手臂弯举",
        "complexity": "simple",
    },
    "push_up": {
        "dir": "fuwo",
        "label": "俯卧撑",
        "complexity": "simple",
    },
    "jumping_jack": {
        "dir": "kaihe",
        "label": "开合跳",
        "complexity": "simple",
    },
    "burpee": {
        "dir": "bobi",
        "label": "波比跳",
        "complexity": "complex",
    },
}

print("运动场景:")
for name, info in EXERCISES.items():
    tag = "[简单]" if info["complexity"] == "simple" else "[复杂]"
    print(f"  {tag} {info['label']} ({name}) -> {info['dir']}/")

## 2. 扫描已有的优化结果

In [ ]:
def find_best_params(scenario_dir: Path) -> list[dict]:
    """在场景目录及其 batch_outputs 子目录中查找所有 best_params JSON."""
    results = []

    # 旧格式: 场景目录根下的 Best_Params_Result_*.json
    for fp in scenario_dir.glob("Best_Params_Result_*.json"):
        with open(fp, "r", encoding="utf-8") as f:
            data = json.load(f)
        stem = fp.stem.replace("Best_Params_Result_", "")
        results.append({"source": str(fp), "sample": stem, **data})

    # 新格式: batch_outputs/batch_runs/<sample>-<mode>-<filter>/<prefix>-best_params.json
    batch_dir = scenario_dir / "batch_outputs" / "batch_runs"
    if batch_dir.is_dir():
        for fp in batch_dir.glob("*/*-best_params.json"):
            with open(fp, "r", encoding="utf-8") as f:
                data = json.load(f)
            parts = fp.parent.name.split("-")
            sample = parts[0] if parts else fp.parent.name
            results.append({"source": str(fp), "sample": sample, **data})

    return results


all_results = {}
for name, info in EXERCISES.items():
    scenario_dir = DATA_DIR / info["dir"]
    if scenario_dir.is_dir():
        params_list = find_best_params(scenario_dir)
        all_results[name] = params_list
        print(f"{info['label']} ({name}): 找到 {len(params_list)} 个优化结果")
    else:
        print(f"{info['label']} ({name}): 目录不存在 - {scenario_dir}")
        all_results[name] = []

## 3. 展示各场景优化效果对比

In [ ]:
rows = []
for name, info in EXERCISES.items():
    for r in all_results.get(name, []):
        rows.append({
            "exercise": info["label"],
            "type": info["complexity"],
            "sample": r.get("sample", "?"),
            "ppg_mode": r.get("ppg_mode", "green"),
            "filter": r.get("adaptive_filter", "lms"),
            "min_err_hf": r.get("min_err_hf", float("nan")),
            "min_err_acc": r.get("min_err_acc", float("nan")),
        })

df_summary = pd.DataFrame(rows)
if not df_summary.empty:
    df_summary = df_summary.sort_values(["type", "exercise", "sample"])
    display(df_summary)
else:
    print("未找到任何优化结果!")

## 4. 构建标准化参数字典

从各场景中选取 HF 融合路径的最优参数 (best_para_hf)，转换为 SolverParams 对象。

In [ ]:
def json_params_to_solver_params(para_dict: dict, ppg_mode: str = "green", adaptive_filter: str = "lms") -> SolverParams:
    """将 JSON 中的 best_para_hf 字典转换为 SolverParams."""
    mapping = {
        "fs_target": "fs_target",
        "max_order": "max_order",
        "spec_penalty_width": "spec_penalty_width",
        "hr_range_hz": "hr_range_hz",
        "slew_limit_bpm": "slew_limit_bpm",
        "slew_step_bpm": "slew_step_bpm",
        "hr_range_rest": "hr_range_rest",
        "slew_limit_rest": "slew_limit_rest",
        "slew_step_rest": "slew_step_rest",
        "smooth_win_len": "smooth_win_len",
        "time_bias": "time_bias",
    }
    kwargs = {}
    for json_key, param_key in mapping.items():
        if json_key in para_dict:
            kwargs[param_key] = para_dict[json_key]
    kwargs["ppg_mode"] = ppg_mode
    kwargs["adaptive_filter"] = adaptive_filter
    return SolverParams(**kwargs)


simple_params = {}
burpee_result = None

for name, info in EXERCISES.items():
    results = all_results.get(name, [])
    if not results:
        status = "!!! 缺失 !!!" if info["complexity"] == "simple" else "(评估目标, 可选)"
        print(f"  {info['label']} ({name}): {status}")
        continue

    best = min(results, key=lambda r: r.get("min_err_hf", float("inf")))
    solver_p = json_params_to_solver_params(
        best["best_para_hf"],
        ppg_mode=best.get("ppg_mode", "green"),
        adaptive_filter=best.get("adaptive_filter", "lms"),
    )

    if info["complexity"] == "simple":
        simple_params[name] = solver_p
        print(f"  {info['label']} ({name}): min_err_hf={best['min_err_hf']:.2f} BPM")
    else:
        burpee_result = best
        print(f"  {info['label']} ({name}): min_err_hf={best['min_err_hf']:.2f} BPM [复杂场景基线]")

## 5. 可行性检查

In [ ]:
missing = [EXERCISES[k]["label"] for k in EXERCISES
           if EXERCISES[k]["complexity"] == "simple" and k not in simple_params]

if missing:
    print("=" * 60)
    print("警告: 以下简单运动场景缺少优化结果:")
    for m in missing:
        print(f"  - {m}")
    print("\n请先通过 GUI 或 CLI 对这些场景运行贝叶斯优化, 然后重新运行本 notebook.")
    print("=" * 60)
else:
    print(f"所有 {len(simple_params)} 个简单运动场景的参数已就绪:")
    for name, p in simple_params.items():
        print(f"  {name}: fs={p.fs_target}Hz, order={p.max_order}, smooth={p.smooth_win_len}")

## 6. 保存参数字典

In [ ]:
output = {
    "simple_params": simple_params,
    "burpee_baseline": burpee_result,
}

output_path = ARTIFACTS_DIR / "simple_params.pkl"
with open(output_path, "wb") as f:
    pickle.dump(output, f)
print(f"已保存到: {output_path}")
print(f"  - simple_params: {list(simple_params.keys())}")
print(f"  - burpee_baseline: {'已包含' if burpee_result else '无'}")